# Phase 1 - Notebook 1b: Per-cluster GIF montages

**Goal.** Qualitative sanity-check on the K-means styles from Notebook 1: for each cluster render a small montage of per-agent GIFs so we can visually confirm the "aggressive" cluster actually contains aggressive-looking driving.

**Inputs:**

- `artifacts/phase1/cleaned_val_with_style.parquet` (or `.csv`) from Notebook 1
- `artifacts/phase1/style_label_map.yaml`

**Outputs:**

- `artifacts/phase1/figures/gifs/<label>_<i>.gif`
- `artifacts/phase1/figures/gifs/manifest.json`

Environment setup and data-path assumptions live in `docs/phase1_gif_setup.md`.

In [ ]:
from __future__ import annotations
import sys, json, yaml
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from tailrisk_mp.runtime import ensure_numpy_pickle_compat
ensure_numpy_pickle_compat()

from tailrisk_mp.style_gifs import render_cluster_montage, render_agent_gif

ARTIFACTS = REPO_ROOT / 'artifacts' / 'phase1'
GIF_DIR   = ARTIFACTS / 'figures' / 'gifs'
GIF_DIR.mkdir(parents=True, exist_ok=True)

VAL_PARQUET = ARTIFACTS / 'cleaned_val_with_style.parquet'
VAL_CSV     = ARTIFACTS / 'cleaned_val_with_style.csv'
LABEL_YAML  = ARTIFACTS / 'style_label_map.yaml'

if VAL_PARQUET.exists():
    val = pd.read_parquet(VAL_PARQUET)
elif VAL_CSV.exists():
    val = pd.read_csv(VAL_CSV)
else:
    raise FileNotFoundError('Run Notebook 1 first to produce cleaned_val_with_style.{parquet,csv}')

label_map = yaml.safe_load(open(LABEL_YAML)) if LABEL_YAML.exists() else {}
print('Loaded val rows:', len(val))
print('Label map:', json.dumps(label_map, indent=2, default=str))
print('Columns of interest:', [c for c in val.columns if c.startswith('style_') or c in ('scenario_id', 'center_objects_id', 'mtr_minfde6')])

## 1. Config

Two sampling modes are rendered for the aggressive cluster:

1. **Random-aggressive** (`mode="random"`): `per_cluster` random agents from the aggressive cluster - catches the *typical* visual signature.
2. **Hardest-aggressive** (`mode="topfde"`): `per_cluster` agents with highest `mtr_minfde6` inside the aggressive cluster - catches the *model-hard* subset we ultimately care about.

Non-aggressive clusters only use the random mode for contrast.

In [ ]:
CFG = {
    'dataset': 'av2',
    'split':   'val',
    'K_for_montage': 2,
    'per_cluster_random':  4,
    'per_cluster_topfde':  4,
    'view_radius': 70.0,
    'frame_stride': 2,
    'fps': 10,
    'max_frames': 40,
    'random_state': 0,
}
CFG

## 2. Render random samples per cluster

In [ ]:
cluster_col = f"style_label_K{CFG['K_for_montage']}"
if cluster_col not in val.columns:
    cluster_col = f"style_cluster_id_K{CFG['K_for_montage']}"
clusters = sorted([c for c in val[cluster_col].dropna().unique()])
print('Rendering for clusters:', clusters, 'column=', cluster_col)

random_paths = render_cluster_montage(
    val,
    cluster_column=cluster_col,
    clusters=clusters,
    per_cluster=CFG['per_cluster_random'],
    dataset=CFG['dataset'],
    split=CFG['split'],
    output_dir=GIF_DIR / 'random',
    view_radius=CFG['view_radius'],
    frame_stride=CFG['frame_stride'],
    fps=CFG['fps'],
    max_frames=CFG['max_frames'],
    title_prefix='random_',
    random_state=CFG['random_state'],
)
print('Rendered random GIFs:', len(random_paths))
for p in random_paths: print(' ', p)

## 3. Render hardest agents inside the aggressive cluster (top-mtr_minfde6)

In [ ]:
topfde_paths = []
if 'mtr_minfde6' in val.columns:
    agg_values = [c for c in clusters if str(c).lower() == 'aggressive']
    if not agg_values:
        agg_values = [clusters[-1]]
    agg_value = agg_values[0]
    aggressive = val[val[cluster_col] == agg_value].copy()
    aggressive = aggressive.dropna(subset=['scenario_id', 'center_objects_id', 'mtr_minfde6'])
    top = aggressive.nlargest(CFG['per_cluster_topfde'], 'mtr_minfde6')
    for i, (_, row) in enumerate(top.iterrows()):
        gif_path = GIF_DIR / 'topfde' / f'aggressive_topfde_{i:02d}.gif'
        gif_path.parent.mkdir(parents=True, exist_ok=True)
        try:
            render_agent_gif(
                dataset=CFG['dataset'], split=CFG['split'],
                scenario_id=str(row['scenario_id']),
                center_objects_id=str(row['center_objects_id']),
                output_path=gif_path,
                title=f"aggressive top-FDE minFDE6={row['mtr_minfde6']:.2f}",
                view_radius=CFG['view_radius'], frame_stride=CFG['frame_stride'],
                fps=CFG['fps'], max_frames=CFG['max_frames'],
            )
            topfde_paths.append(gif_path)
        except Exception as err:
            print(f"skip {row['scenario_id']}: {err}")
    print('Rendered top-minFDE GIFs:', len(topfde_paths))
else:
    print('mtr_minfde6 not in val columns; skip top-FDE montage.')
for p in topfde_paths: print(' ', p)

## 4. Inline preview + manifest

Preview a few GIFs inside the notebook and write a JSON manifest (path -> per-row metadata) so the report can cite specific scenarios.

In [ ]:
from IPython.display import Image, display

def _preview(paths, n=4):
    for p in list(paths)[:n]:
        print(p)
        try:
            display(Image(filename=str(p)))
        except Exception as err:
            print('  (preview failed:', err, ')')

print('=== random samples ==='); _preview(random_paths, n=min(6, len(random_paths)))
print('\n=== top-FDE aggressive ==='); _preview(topfde_paths, n=min(4, len(topfde_paths)))

manifest = {
    'config': CFG,
    'cluster_column': cluster_col,
    'random': [str(p) for p in random_paths],
    'topfde': [str(p) for p in topfde_paths],
}
with open(GIF_DIR / 'manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2, default=str)
print('\nWrote', GIF_DIR / 'manifest.json')